# Phase 6 Validation: Restoration Candidates

This notebook validates the outputs generated by src/optimization/candidate_generator.py.
It displays:
1. Top 10 most expensive candidates by cost_cj
2. Top 10 highest aggregate marginal gains by aggregate_delta_expected


In [ ]:
from pathlib import Path
import pandas as pd

candidate_path = Path("data/processed/candidate_library.csv")
delta_path = Path("data/processed/marginal_gains_delta.csv")

if not candidate_path.exists() or not delta_path.exists():
    candidate_path = Path("../data/processed/candidate_library.csv").resolve()
    delta_path = Path("../data/processed/marginal_gains_delta.csv").resolve()

print(f"Candidate file: {candidate_path}")
print(f"Delta file: {delta_path}")
print(f"Candidate exists: {candidate_path.exists()}")
print(f"Delta exists: {delta_path.exists()}")

In [ ]:
candidates = pd.read_csv(candidate_path)
delta = pd.read_csv(delta_path)

print("Candidate rows:", len(candidates))
print("Delta rows:", len(delta))
print("Candidate columns:", sorted(candidates.columns.tolist()))
print("Delta columns:", sorted(delta.columns.tolist()))

In [ ]:
top_cost = candidates.nlargest(10, "cost_cj")[
    ["candidate_id", "route_id", "direction_id", "time_period", "increment_trips", "cost_cj"]
]
top_cost

In [ ]:
top_gain = candidates.nlargest(10, "aggregate_delta_expected")[
    [
        "candidate_id",
        "route_id",
        "direction_id",
        "time_period",
        "increment_trips",
        "aggregate_delta_expected",
    ]
]
top_gain

In [ ]:
delta_agg = (
    delta.groupby("candidate_j", as_index=False)["delta_score"]
    .sum()
    .rename(columns={"candidate_j": "candidate_id", "delta_score": "aggregate_delta_from_delta_csv"})
)

crosscheck = (
    candidates[["candidate_id", "aggregate_delta_raw", "aggregate_delta_expected"]]
    .merge(delta_agg, on="candidate_id", how="left")
    .fillna({"aggregate_delta_from_delta_csv": 0.0})
)

crosscheck["abs_diff_raw_vs_delta"] = (
    crosscheck["aggregate_delta_raw"] - crosscheck["aggregate_delta_from_delta_csv"]
).abs()

crosscheck.sort_values("abs_diff_raw_vs_delta", ascending=False).head(10)